In [1]:
import os
import sys
import gc
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, balanced_accuracy_score
)
from sklearn.utils.class_weight import compute_class_weight

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
PROJECT_ROOT = r"C:\Users\hp\StressDetection"
DL_PATH      = os.path.join(PROJECT_ROOT, "DL", "outputttt")
SAVE_DIR     = os.path.join(PROJECT_ROOT, "DL")

if os.path.join(PROJECT_ROOT, "DL") not in sys.path:
    sys.path.insert(0, os.path.join(PROJECT_ROOT, "DL"))

from data_loader import load_windows

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow version:", tf.__version__)

X, y, pids = load_windows(DL_PATH, file_type="mtf")

print("\nFinal shapes:")
print("X:", X.shape)
print("y:", y.shape)
print("Participants:", len(np.unique(pids)))

LOADING file_type='mtf'  tasks=['T1', 'T2', 'T3']
Source: C:\Users\hp\StressDetection\DL\outputttt
Found 56 participant folders
----------------------------------------------------------------


TensorFlow version: 2.20.0


----------------------------------------------------------------
Files loaded   : 168
Files missing  : 0
Total windows  : 29735
Window shape   : (64, 64, 5)
Label 0 (calm) : 9911
Label 1 (stress): 19824
Class balance  : 2.00 : 1  (stress : calm)
Participants   : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56]



Final shapes:
X: (29735, 64, 64, 5)
y: (29735,)
Participants: 56


In [2]:
def build_cnn(input_shape):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),

        tf.keras.layers.Conv2D(16, 3, padding="same", activation="relu"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D(),

        tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPooling2D(),

        tf.keras.layers.Conv2D(64, 3, padding="same", activation="relu"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.GlobalAveragePooling2D(),

        tf.keras.layers.Dropout(0.35),
        tf.keras.layers.Dense(32, activation="relu"),
        tf.keras.layers.Dropout(0.25),

        tf.keras.layers.Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model


def scale_mtf(X_train, X_val, X_test):
    X_train = X_train.astype(np.float32)
    X_val   = X_val.astype(np.float32)
    X_test  = X_test.astype(np.float32)

    if X_train.max() > 1.5:
        X_train /= 255.0
        X_val   /= 255.0
        X_test  /= 255.0

    return X_train, X_val, X_test

In [3]:
EPOCHS     = 20
BATCH_SIZE = 32
MAX_FOLDS  = None
SEED       = 42

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

cnn_results = []   # fold-level metrics (same as before)
all_probs   = []   # ← NEW: per-window probabilities for fusion

unique_pids = sorted(np.unique(pids))

for fold_number, held_out_pid in enumerate(unique_pids, start=1):

    if MAX_FOLDS is not None and fold_number > MAX_FOLDS:
        break

    fold_seed = SEED + int(held_out_pid)

    print(f"\n{'=' * 65}")
    print(
        f"LOSO CNN — Fold {fold_number}/{len(unique_pids)} | "
        f"Test participant: p{held_out_pid}"
    )
    print(f"{'=' * 65}")

    # -------------------------------
    # OUTER LOSO SPLIT
    # -------------------------------
    test_mask  = (pids == held_out_pid)
    train_mask = ~test_mask

    X_outer_train  = X[train_mask]
    y_outer_train  = y[train_mask]
    pids_outer_train = pids[train_mask]

    X_test = X[test_mask]
    y_test = y[test_mask]

    # -------------------------------
    # INNER TRAIN / VALIDATION SPLIT
    # -------------------------------
    inner_splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=0.10,
        random_state=fold_seed
    )

    train_idx, val_idx = next(
        inner_splitter.split(
            X_outer_train,
            y_outer_train,
            groups=pids_outer_train
        )
    )

    X_train = X_outer_train[train_idx]
    y_train = y_outer_train[train_idx]

    X_val = X_outer_train[val_idx]
    y_val = y_outer_train[val_idx]

    # -------------------------------
    # SCALE USING TRAIN DATA ONLY
    # -------------------------------
    X_train, X_val, X_test_scaled = scale_mtf(X_train, X_val, X_test)

    # -------------------------------
    # CLASS WEIGHTS
    # -------------------------------
    classes = np.unique(y_train)
    weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=y_train
    )
    class_weights = {
        int(label): float(weight)
        for label, weight in zip(classes, weights)
    }
    print("Class weights:", class_weights)

    # -------------------------------
    # CREATE MODEL
    # -------------------------------
    tf.keras.backend.clear_session()
    gc.collect()
    tf.keras.utils.set_random_seed(fold_seed)

    model = build_cnn(X_train.shape[1:])

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=4,
            restore_best_weights=True,
            verbose=1
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            min_lr=1e-5,
            verbose=1
        )
    ]

    # -------------------------------
    # TRAIN
    # -------------------------------
    model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        class_weight=class_weights,
        callbacks=callbacks,
        verbose=0
    )

    # -------------------------------
    # SELECT THRESHOLD ON VALIDATION
    # -------------------------------
    val_probabilities = model.predict(X_val, verbose=0).ravel()

    thresholds       = np.arange(0.30, 0.81, 0.01)
    best_threshold   = 0.50
    best_val_bal_acc = -1

    for threshold in thresholds:
        val_predictions = (val_probabilities >= threshold).astype(int)
        val_bal_acc     = balanced_accuracy_score(y_val, val_predictions)
        if val_bal_acc > best_val_bal_acc:
            best_val_bal_acc = val_bal_acc
            best_threshold   = threshold

    print(
        f"Best validation threshold: {best_threshold:.2f} | "
        f"Validation Balanced Accuracy: {best_val_bal_acc:.4f}"
    )

    # -------------------------------
    # TEST ON HELD-OUT PARTICIPANT
    # -------------------------------
    test_probabilities = model.predict(X_test_scaled, verbose=0).ravel()
    predictions        = (test_probabilities >= best_threshold).astype(int)

    # ← NEW: collect per-window probabilities for fusion
    for prob, true_label in zip(test_probabilities, y_test):
        all_probs.append({
            "pid":        held_out_pid,
            "true_label": int(true_label),
            "prob_stress": float(prob)
        })

    fold_result = {
        "participant":   held_out_pid,
        "threshold":     best_threshold,
        "accuracy":      accuracy_score(y_test, predictions),
        "f1":            f1_score(y_test, predictions, zero_division=0),
        "precision":     precision_score(y_test, predictions, zero_division=0),
        "recall":        recall_score(y_test, predictions, zero_division=0),
        "balanced_acc":  balanced_accuracy_score(y_test, predictions),
        "n_test_windows": len(y_test)
    }

    cnn_results.append(fold_result)

    print(
        f"p{held_out_pid} | "
        f"Threshold={best_threshold:.2f} | "
        f"Accuracy={fold_result['accuracy'] * 100:.2f}% | "
        f"F1={fold_result['f1']:.4f} | "
        f"Bal.Acc={fold_result['balanced_acc']:.4f}"
    )

# -------------------------------
# FINAL RESULTS
# -------------------------------
cnn_results_df = pd.DataFrame(cnn_results)

print("\n" + "=" * 65)
print("CNN MTF — LOSO RESULTS")
print("=" * 65)
print(f"Folds completed: {len(cnn_results_df)}")
print(f"Mean Accuracy:   {cnn_results_df['accuracy'].mean() * 100:.2f}%")
print(f"Mean F1:         {cnn_results_df['f1'].mean():.4f}")
print(f"Mean Precision:  {cnn_results_df['precision'].mean():.4f}")
print(f"Mean Recall:     {cnn_results_df['recall'].mean():.4f}")
print(f"Mean Bal. Acc:   {cnn_results_df['balanced_acc'].mean():.4f}")

# ← NEW: save fold-level metrics
metrics_path = os.path.join(SAVE_DIR, "cnn_loso_metrics.csv")
cnn_results_df.to_csv(metrics_path, index=False)
print(f"\nMetrics saved  → {metrics_path}")

# ← NEW: save per-window probabilities for fusion
probs_df   = pd.DataFrame(all_probs)
probs_path = os.path.join(SAVE_DIR, "cnn_probs.csv")
probs_df.to_csv(probs_path, index=False)
print(f"Probabilities saved → {probs_path}")
print(f"Total windows saved: {len(probs_df)}")
print()
probs_df.head(10)


LOSO CNN — Fold 1/56 | Test participant: p1
Class weights: {0: 1.5001153136531364, 1: 0.7499711749106422}


Epoch 8: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.

Epoch 10: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.
Epoch 10: early stopping
Restoring model weights from the end of the best epoch: 6.
Best validation threshold: 0.70 | Validation Balanced Accuracy: 0.9237
p1 | Threshold=0.70 | Accuracy=80.60% | F1=0.8681 | Bal.Acc=0.7302

LOSO CNN — Fold 2/56 | Test participant: p2
Class weights: {0: 1.5001153136531364, 1: 0.7499711749106422}

Epoch 5: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.

Epoch 7: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.
Epoch 7: early stopping
Restoring model weights from the end of the best epoch: 3.
Best validation threshold: 0.80 | Validation Balanced Accuracy: 0.9223
p2 | Threshold=0.80 | Accuracy=85.50% | F1=0.8780 | Bal.Acc=0.8912

LOSO CNN — Fold 3/56 | Test part

,pid,true_label,prob_stress
0,1,0,0.910990
1,1,0,0.948127
2,1,0,0.820668
3,1,0,0.976522
4,1,0,0.970606
5,1,0,0.912416
6,1,0,0.872614
7,1,0,0.492368
8,1,0,0.458340
9,1,0,0.498718
